# EE 451: Communications Systems
## Lesson 24 - Random Variables & Gaussian Distribution

### Learning Objectives
By the end of this lesson, you will be able to:
- Distinguish between discrete and continuous random variables
- Define and calculate PMF and PDF
- Use Cumulative Distribution Functions (CDF)
- Analyze Gaussian distribution properties (68-95-99.7 rule)
- Apply the Q-function to tail probability calculations
- Use the inverse Q-function to find required Eb/N0 for a target BER

### Textbook Reference
Haykin & Moher, Chapter 8.3-8.4

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erfc
from scipy.stats import binom
import warnings
warnings.filterwarnings('ignore')

# Use consistent plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Discrete Random Variables

**Discrete RV:** Takes countable values $x_1, x_2, \ldots$

**Probability Mass Function (PMF):** $P_X(x) = P(X = x)$
- $P_X(x) \geq 0$ for all $x$
- $\sum_x P_X(x) = 1$

**Expected Value:** $E[X] = \sum_x x \cdot P_X(x)$

**Variance:** $\text{Var}(X) = E[(X - \mu)^2] = E[X^2] - (E[X])^2$

**Example:** Bit errors in an $N$-bit packet follow a **Binomial distribution**

In [ ]:
# Part 1: Discrete Random Variables - PMF

# X = number of bit errors in N=10 bits, p_error = 0.1
N_bits = 10
p_err = 0.1

x_values = np.arange(0, N_bits + 1)
pmf_values = binom.pmf(x_values, N_bits, p_err)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: PMF bar chart ---
ax = axes[0]
bars = ax.bar(x_values, pmf_values, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Bit Errors (k)')
ax.set_ylabel('P(X = k)')
ax.set_title(f'PMF: Bit Errors in {N_bits}-bit Packet (p = {p_err})')
ax.set_xticks(x_values)

# Highlight the mean
E_X = N_bits * p_err
ax.axvline(x=E_X, color='red', linestyle='--', linewidth=2, label=f'E[X] = {E_X:.1f}')
ax.legend()

# --- Plot 2: Compare different error probabilities ---
ax = axes[1]
p_values = [0.01, 0.05, 0.1, 0.3]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(p_values)))

for p, color in zip(p_values, colors):
    pmf = binom.pmf(x_values, N_bits, p)
    ax.plot(x_values, pmf, 'o-', color=color, linewidth=2, markersize=6,
            label=f'p = {p} (E[X]={N_bits*p:.1f})')

ax.set_xlabel('Number of Bit Errors (k)')
ax.set_ylabel('P(X = k)')
ax.set_title(f'PMF for Different Error Probabilities (N = {N_bits})')
ax.set_xticks(x_values)
ax.legend()

plt.tight_layout()
plt.show()

# Print statistics
print("=" * 55)
print("Discrete RV Statistics: Bit Errors in 10-bit Packet")
print("=" * 55)
print(f"\nBinomial(N={N_bits}, p={p_err})")
Var_X = N_bits * p_err * (1 - p_err)
print(f"  Expected value: E[X] = Np = {E_X:.2f}")
print(f"  Variance:     Var(X) = Np(1-p) = {Var_X:.2f}")
print(f"  Std deviation:   sigma = {np.sqrt(Var_X):.4f}")
print(f"\nPMF values:")
for k in range(6):
    print(f"  P(X = {k}) = {binom.pmf(k, N_bits, p_err):.6f}")

## Part 2: Continuous Random Variables

**Continuous RV:** Takes values in an interval (uncountable)

**Probability Density Function (PDF):** $f_X(x)$
- $f_X(x) \geq 0$ for all $x$
- $\int_{-\infty}^{\infty} f_X(x) \, dx = 1$
- $P(a < X < b) = \int_a^b f_X(x) \, dx$
- Note: $P(X = x) = 0$ for any specific value

**Key Distributions:**
- **Uniform:** Equal probability over an interval
- **Exponential:** Models inter-arrival times
- **Gaussian:** The most important distribution in communications!

In [ ]:
# Part 2: Continuous Random Variables - PDF

x = np.linspace(-1, 6, 1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Uniform distribution ---
ax = axes[0]
a, b = 0, 3
pdf_uniform = np.where((x >= a) & (x <= b), 1/(b - a), 0)
ax.plot(x, pdf_uniform, 'b-', linewidth=2)
ax.fill_between(x, pdf_uniform, alpha=0.3, color='steelblue')

# Shade P(1 < X < 2)
mask = (x >= 1) & (x <= 2)
ax.fill_between(x[mask], pdf_uniform[mask], alpha=0.6, color='coral',
                label=f'P(1 < X < 2) = {1/(b-a):.3f}')
ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title(f'Uniform Distribution [{a}, {b}]')
ax.set_ylim(-0.05, 0.6)
ax.legend()

# --- Plot 2: Exponential distribution ---
ax = axes[1]
lambdas = [0.5, 1.0, 2.0]
x_exp = np.linspace(0, 6, 500)
exp_colors = ['steelblue', 'coral', 'green']

for lam, color in zip(lambdas, exp_colors):
    pdf_exp = stats.expon.pdf(x_exp, scale=1/lam)
    ax.plot(x_exp, pdf_exp, color=color, linewidth=2,
            label=f'\u03bb = {lam} (E[X] = {1/lam:.1f})')

ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title('Exponential Distribution')
ax.legend()
ax.set_ylim(0, 2.2)

# --- Plot 3: PDF property — area = probability ---
ax = axes[2]
mu, sigma = 2, 0.8
pdf_gauss = stats.norm.pdf(x, mu, sigma)
ax.plot(x, pdf_gauss, 'b-', linewidth=2)
ax.fill_between(x, pdf_gauss, alpha=0.3, color='steelblue', label='Total area = 1')

# Shade P(X > 3)
mask = x >= 3
area = 1 - stats.norm.cdf(3, mu, sigma)
ax.fill_between(x[mask], pdf_gauss[mask], alpha=0.6, color='coral',
                label=f'P(X > 3) = {area:.4f}')
ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title(f'Gaussian(\u03bc={mu}, \u03c3={sigma}): Area = Probability')
ax.legend()

plt.tight_layout()
plt.show()

# Print key formulas
print("=" * 55)
print("Continuous RV Properties")
print("=" * 55)
print(f"\nUniform[{a},{b}]: E[X] = (a+b)/2 = {(a+b)/2}, Var(X) = (b-a)^2/12 = {(b-a)**2/12:.4f}")
print(f"Exponential(lam=1): E[X] = 1/lam = 1.0, Var(X) = 1/lam^2 = 1.0")
print(f"Gaussian(mu={mu}, sigma={sigma}): P(X > 3) = {area:.4f}")

## Part 3: Gaussian (Normal) Distribution

**PDF:** $f_X(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$

**Parameters:** $\mu$ (mean), $\sigma^2$ (variance)

**Standard Gaussian:** $\mu = 0$, $\sigma^2 = 1$

**68-95-99.7 Rule:**
- 68.3% of values within $\mu \pm \sigma$
- 95.4% within $\mu \pm 2\sigma$
- 99.7% within $\mu \pm 3\sigma$

**Why Gaussian?**
- Central Limit Theorem: sum of many independent RVs approaches Gaussian
- Thermal noise in receivers is Gaussian (AWGN)
- Makes BER analysis tractable

In [ ]:
# Part 3: Gaussian Distribution

x = np.linspace(-6, 10, 1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Different mu and sigma ---
ax = axes[0]
params = [(0, 1, 'N(0,1)'), (2, 1, 'N(2,1)'), (0, 2, 'N(0,4)'), (2, 0.5, 'N(2,0.25)')]
g_colors = ['steelblue', 'coral', 'green', 'mediumpurple']

for (mu, sigma, label), color in zip(params, g_colors):
    pdf = stats.norm.pdf(x, mu, sigma)
    ax.plot(x, pdf, color=color, linewidth=2, label=label)

ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title('Gaussian PDF: Effect of \u03bc and \u03c3')
ax.legend()
ax.set_xlim(-6, 8)

# --- Plot 2: 68-95-99.7 Rule ---
ax = axes[1]
mu, sigma = 0, 1
pdf = stats.norm.pdf(x, mu, sigma)
ax.plot(x, pdf, 'k-', linewidth=2)

sigmas = [(1, 'lightskyblue', '68.3%'), (2, 'lightblue', '95.4%'), (3, 'lavender', '99.7%')]
for n_sigma, color, pct in reversed(sigmas):
    mask = (x >= mu - n_sigma * sigma) & (x <= mu + n_sigma * sigma)
    ax.fill_between(x[mask], pdf[mask], alpha=0.8, color=color,
                    label=f'\u03bc \u00b1 {n_sigma}\u03c3: {pct}')

ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title('68-95-99.7 Rule (Standard Gaussian)')
ax.legend(fontsize=10)
ax.set_xlim(-4.5, 4.5)

# --- Plot 3: AWGN histogram vs theoretical PDF ---
ax = axes[2]
np.random.seed(42)
N_samples = 50000
noise_sigma = 1.5
noise_samples = np.random.normal(0, noise_sigma, N_samples)

ax.hist(noise_samples, bins=100, density=True, alpha=0.6, color='steelblue',
        edgecolor='white', label=f'AWGN samples (N={N_samples:,})')
x_plot = np.linspace(-6, 6, 500)
ax.plot(x_plot, stats.norm.pdf(x_plot, 0, noise_sigma), 'r-', linewidth=2,
        label=f'N(0, {noise_sigma}\u00b2) theoretical')
ax.set_xlabel('Amplitude')
ax.set_ylabel('Probability Density')
ax.set_title('AWGN: Histogram vs Gaussian PDF')
ax.legend()

plt.tight_layout()
plt.show()

# Print statistics
print("=" * 55)
print("Gaussian Distribution Properties")
print("=" * 55)
print(f"\nAWGN Samples (N = {N_samples:,}, sigma = {noise_sigma}):")
print(f"  Sample mean:  {np.mean(noise_samples):.4f}  (theory: 0)")
print(f"  Sample std:   {np.std(noise_samples):.4f}  (theory: {noise_sigma})")
print(f"  Sample var:   {np.var(noise_samples):.4f}  (theory: {noise_sigma**2})")
print(f"\n68-95-99.7 Rule Verification:")
for n in [1, 2, 3]:
    in_range = np.mean(np.abs(noise_samples) < n * noise_sigma)
    theoretical = stats.norm.cdf(n) - stats.norm.cdf(-n)
    print(f"  Within +/-{n}sigma: {in_range:.4f} (theory: {theoretical:.4f})")

## Part 4: CDF and Standardization

**CDF:** $F_X(x) = P(X \leq x) = \int_{-\infty}^x f_X(t) \, dt$

**Properties:**
- $0 \leq F_X(x) \leq 1$
- $F_X(-\infty) = 0$, $F_X(\infty) = 1$
- Non-decreasing
- $f_X(x) = dF_X(x)/dx$

**Standardization:** Convert any Gaussian to standard form
$$Z = \frac{X - \mu}{\sigma} \sim N(0, 1)$$

Then: $P(X > a) = P\left(Z > \frac{a - \mu}{\sigma}\right) = Q\left(\frac{a - \mu}{\sigma}\right)$

In [ ]:
# Part 4: CDF and Standardization

x = np.linspace(-5, 5, 1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: PDF and CDF side by side ---
ax = axes[0]
mu, sigma = 0, 1
pdf = stats.norm.pdf(x, mu, sigma)
cdf = stats.norm.cdf(x, mu, sigma)

ax.plot(x, pdf, 'b-', linewidth=2, label='PDF f_X(x)')
ax.plot(x, cdf, 'r-', linewidth=2, label='CDF F_X(x)')
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('x')
ax.set_ylabel('Value')
ax.set_title('Standard Gaussian: PDF and CDF')
ax.legend()

# --- Plot 2: Standardization example ---
ax = axes[1]
mu_orig, sigma_orig = 5, 2
x_orig = np.linspace(-3, 13, 500)
pdf_orig = stats.norm.pdf(x_orig, mu_orig, sigma_orig)

ax.plot(x_orig, pdf_orig, 'b-', linewidth=2,
        label=f'X ~ N({mu_orig}, {sigma_orig}\u00b2)')
ax.fill_between(x_orig, pdf_orig, where=(x_orig > 7), alpha=0.4, color='coral')

# Annotate
P_gt7 = 1 - stats.norm.cdf(7, mu_orig, sigma_orig)
z_score = (7 - mu_orig) / sigma_orig
ax.axvline(x=7, color='red', linestyle='--', linewidth=1.5)
ax.text(8.5, 0.12, f'P(X > 7)\n= P(Z > {z_score:.0f})\n= Q({z_score:.0f})\n= {P_gt7:.4f}',
        fontsize=11, color='red',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title('Standardization: Z = (X \u2212 \u03bc) / \u03c3')
ax.legend()

plt.tight_layout()
plt.show()

# Print worked example
print("=" * 55)
print("Standardization Example")
print("=" * 55)
print(f"\nX ~ N(mu={mu_orig}, sigma^2={sigma_orig**2})")
print(f"Find: P(X > 7)")
print(f"\nStep 1: Standardize")
print(f"  Z = (X - mu) / sigma = (7 - {mu_orig}) / {sigma_orig} = {z_score:.1f}")
print(f"\nStep 2: Use Q-function")
print(f"  P(X > 7) = P(Z > {z_score:.1f}) = Q({z_score:.1f}) = {P_gt7:.4f}")
print(f"\nVerification: scipy.stats.norm.sf(7, {mu_orig}, {sigma_orig}) = {stats.norm.sf(7, mu_orig, sigma_orig):.4f}")

## Part 5: The Q-Function

**Definition:** Tail probability of the standard Gaussian
$$Q(x) = P(Z > x) = \frac{1}{\sqrt{2\pi}} \int_x^{\infty} e^{-t^2/2} \, dt$$

**Properties:**
- $Q(0) = 0.5$
- $Q(-x) = 1 - Q(x)$ (symmetry)
- $Q(x) \to 0$ as $x \to \infty$

**Relationship to erfc:**
$$Q(x) = \frac{1}{2} \text{erfc}\left(\frac{x}{\sqrt{2}}\right)$$

**Why Q-function matters:** BER for BPSK is $P_e = Q\left(\sqrt{\frac{2E_b}{N_0}}\right)$

In [ ]:
# Part 5: Q-Function

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Q-function on log scale ---
ax = axes[0]
x_q = np.linspace(0, 6, 500)
Q_values = stats.norm.sf(x_q)  # sf = survival function = Q(x)

ax.semilogy(x_q, Q_values, 'b-', linewidth=2)

# Mark common values
common_x = [0, 1, 2, 3, 4, 5]
common_Q = [stats.norm.sf(xi) for xi in common_x]
ax.plot(common_x, common_Q, 'ro', markersize=8)

for xi, qi in zip(common_x, common_Q):
    if xi == 0:
        ax.annotate(f'Q({xi}) = {qi:.2f}', xy=(xi, qi),
                    xytext=(xi + 0.3, qi * 0.3), fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='red'))
    else:
        ax.annotate(f'Q({xi}) = {qi:.2e}', xy=(xi, qi),
                    xytext=(xi + 0.2, qi * 5), fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('x')
ax.set_ylabel('Q(x)')
ax.set_title('Q-Function (log scale)')
ax.set_ylim(1e-10, 1)

# --- Plot 2: Q-function vs erfc relationship ---
ax = axes[1]
Q_from_sf = stats.norm.sf(x_q)
Q_from_erfc = 0.5 * erfc(x_q / np.sqrt(2))

ax.semilogy(x_q, Q_from_sf, 'b-', linewidth=2, label='Q(x) = 1 - \u03a6(x)')
ax.semilogy(x_q, Q_from_erfc, 'r--', linewidth=2, label='Q(x) = \u00bd erfc(x/\u221a2)')
ax.set_xlabel('x')
ax.set_ylabel('Q(x)')
ax.set_title('Q-Function: Two Equivalent Forms')
ax.legend()
ax.set_ylim(1e-10, 1)

plt.tight_layout()
plt.show()

# Print Q-function table
print("=" * 50)
print("Q-Function Reference Table")
print("=" * 50)
print(f"{'x':>6}  {'Q(x)':>14}")
print("-" * 22)
for xi in [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6]:
    qi = stats.norm.sf(xi)
    if qi > 0.01:
        print(f"{xi:>6.1f}  {qi:>14.6f}")
    else:
        print(f"{xi:>6.1f}  {qi:>14.2e}")
print(f"\nKey: Q(x) = (1/2) erfc(x/sqrt(2))")
print(f"     Q(-x) = 1 - Q(x)")

## Part 6: BER Application — BPSK Performance

**BPSK BER:** $P_e = Q\left(\sqrt{\frac{2E_b}{N_0}}\right)$

**Inverse Q-Function:** Given target BER, find required $E_b/N_0$

**Procedure:**
1. Set $Q(x) = P_e^{\text{target}}$
2. Find $x = Q^{-1}(P_e)$ using table or `scipy.stats.norm.ppf(1 - P_e)`
3. Solve: $\sqrt{2 E_b/N_0} = x \implies E_b/N_0 = x^2/2$
4. Convert to dB: $(E_b/N_0)_{\text{dB}} = 10 \log_{10}(x^2/2)$

In [ ]:
# Part 6: BER Application - BPSK Performance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: BPSK BER curve ---
ax = axes[0]
EbN0_dB = np.linspace(-2, 14, 200)
EbN0_linear = 10 ** (EbN0_dB / 10)
BER_BPSK = stats.norm.sf(np.sqrt(2 * EbN0_linear))

ax.semilogy(EbN0_dB, BER_BPSK, 'b-', linewidth=2, label='BPSK: Q(\u221a(2Eb/N\u2080))')

# Mark key BER targets
targets = [(1e-3, 'BER = 10\u207b\u00b3'), (1e-5, 'BER = 10\u207b\u2075'),
           (1e-6, 'BER = 10\u207b\u2076')]
for target, label in targets:
    x_req = stats.norm.ppf(1 - target)  # Inverse Q-function
    EbN0_req = x_req**2 / 2
    EbN0_req_dB = 10 * np.log10(EbN0_req)
    ax.plot(EbN0_req_dB, target, 'ro', markersize=8)
    ax.annotate(f'{label}\nEb/N\u2080 = {EbN0_req_dB:.1f} dB',
                xy=(EbN0_req_dB, target), xytext=(EbN0_req_dB - 4, target * 15),
                fontsize=9, arrowprops=dict(arrowstyle='->', color='red'),
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Eb/N\u2080 (dB)')
ax.set_ylabel('Bit Error Rate (BER)')
ax.set_title('BPSK BER Performance')
ax.set_ylim(1e-8, 1)
ax.set_xlim(-2, 14)
ax.legend()

# --- Plot 2: Required Eb/N0 for target BER ---
ax = axes[1]
target_BER = np.logspace(-7, -1, 100)
x_values = stats.norm.ppf(1 - target_BER)
EbN0_required_dB = 10 * np.log10(x_values**2 / 2)

ax.semilogx(target_BER, EbN0_required_dB, 'b-', linewidth=2)
ax.set_xlabel('Target BER')
ax.set_ylabel('Required Eb/N\u2080 (dB)')
ax.set_title('BPSK: Required Eb/N\u2080 for Target BER')

for target, label in targets:
    x_req = stats.norm.ppf(1 - target)
    EbN0_dB_val = 10 * np.log10(x_req**2 / 2)
    ax.plot(target, EbN0_dB_val, 'ro', markersize=8)
    ax.annotate(f'{EbN0_dB_val:.1f} dB', xy=(target, EbN0_dB_val),
                xytext=(target * 5, EbN0_dB_val + 0.5), fontsize=10)

ax.invert_xaxis()

plt.tight_layout()
plt.show()

# Worked example: Find Eb/N0 for BER = 10^-6
print("=" * 60)
print("Worked Example: BPSK  Find Eb/N0 for BER = 1e-6")
print("=" * 60)
target = 1e-6
print(f"\nGiven: P_e = Q(sqrt(2*Eb/N0)) = {target}")

x = stats.norm.ppf(1 - target)
print(f"\nStep 1: Inverse Q-function")
print(f"  Q(x) = {target} --> x = Q_inv({target}) = {x:.4f}")
print(f"  (Python: norm.ppf(1 - {target}) = {x:.4f})")

EbN0 = x**2 / 2
EbN0_dB_val = 10 * np.log10(EbN0)
print(f"\nStep 2: Solve for Eb/N0")
print(f"  sqrt(2*Eb/N0) = {x:.4f}")
print(f"  2*Eb/N0 = {x**2:.4f}")
print(f"  Eb/N0 = {EbN0:.4f} = {EbN0_dB_val:.2f} dB")
print(f"\nVerification: Q(sqrt(2 * {EbN0:.4f})) = Q({x:.4f}) = {stats.norm.sf(x):.2e}")

## Summary

### Key Formulas

| Concept | Formula |
|---------|--------|
| PMF (discrete) | $P_X(x) = P(X = x)$, $\sum P_X(x) = 1$ |
| PDF (continuous) | $P(a < X < b) = \int_a^b f_X(x)\,dx$ |
| CDF | $F_X(x) = P(X \leq x)$ |
| Expected value | $E[X] = \sum x P_X(x)$ or $\int x f_X(x)\,dx$ |
| Variance | $\text{Var}(X) = E[X^2] - (E[X])^2$ |
| Gaussian PDF | $f_X(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-(x-\mu)^2/(2\sigma^2)}$ |
| Standardization | $Z = (X - \mu)/\sigma$ |
| Q-function | $Q(x) = \frac{1}{2}\text{erfc}(x/\sqrt{2})$ |
| BPSK BER | $P_e = Q\left(\sqrt{2E_b/N_0}\right)$ |

### Q-Function Quick Reference

| $x$ | $Q(x)$ | Common Usage |
|-----|--------|-------------|
| 0 | 0.5 | |
| 1 | 0.159 | $\mu \pm \sigma$ boundary |
| 2 | 0.0228 | $\mu \pm 2\sigma$ boundary |
| 3 | 1.35e-3 | $\mu \pm 3\sigma$ boundary |
| 4.75 | 1e-6 | BPSK at 10.5 dB |

### Key Takeaways
1. **Discrete RVs** have PMFs; **continuous RVs** have PDFs
2. **Gaussian distribution** models AWGN — the dominant noise in communications
3. **Standardization** converts any Gaussian to standard form for Q-function lookup
4. **Q-function** gives tail probabilities — directly used in BER calculations
5. **Inverse Q-function** determines the SNR needed to achieve a target BER

### Next Topics
- **Lesson 25:** Noise fundamentals, thermal noise, noise figure
- **Lesson 27:** Full BER derivation, matched filtering